# 02. LightGBM 오차 분석

전체 점수에서 출발해 날짜·상품군·점포별 실패 구간을 확인합니다.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_PATH = PROJECT_ROOT / "outputs"

In [ ]:
def add_errors(frame):
    frame = frame.copy()
    frame["residual"] = frame["target_sales"] - frame["prediction"]
    frame["absolute_error"] = frame["residual"].abs()
    frame["squared_log_error"] = (
        np.log1p(frame["target_sales"].clip(lower=0))
        - np.log1p(frame["prediction"].clip(lower=0))
    ) ** 2
    return frame


def group_scores(frame, column):
    return (
        frame.groupby(column, observed=True)
        .agg(
            rows=("target_sales", "size"),
            actual_mean=("target_sales", "mean"),
            prediction_mean=("prediction", "mean"),
            mae=("absolute_error", "mean"),
            mean_squared_log_error=("squared_log_error", "mean"),
        )
        .assign(rmsle=lambda x: np.sqrt(x["mean_squared_log_error"]))
        .drop(columns="mean_squared_log_error")
        .sort_values("rmsle", ascending=False)
    )


def top_errors(frame, count=20):
    columns = ["target_date", "store_nbr", "family", "target_sales", "prediction", "residual", "absolute_error"]
    return frame.nlargest(count, "absolute_error")[columns]

In [ ]:
with (OUTPUT_PATH / "metrics.json").open(encoding="utf-8") as file:
    metrics = json.load(file)

predictions = pd.read_csv(OUTPUT_PATH / "validation_predictions.csv", parse_dates=["target_date"])
importance = pd.read_csv(OUTPUT_PATH / "feature_importance.csv")
predictions = add_errors(predictions)

pd.DataFrame([metrics]).T.rename(columns={0: "value"})

## 실제값과 예측값

고판매 구간의 영향을 줄여 전체 분포를 보기 위해 로그 축을 사용합니다.

In [ ]:
sample = predictions.sample(min(30_000, len(predictions)), random_state=42)
limit = max(sample["target_sales"].max(), sample["prediction"].max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(sample["target_sales"], sample["prediction"], alpha=0.15, s=8)
axes[0].plot([0, limit], [0, limit], linestyle="--", color="black", linewidth=1)
axes[0].set(xscale="symlog", yscale="symlog", title="Actual vs predicted", xlabel="actual", ylabel="predicted")
axes[1].hist(predictions["residual"].clip(predictions["residual"].quantile(0.01), predictions["residual"].quantile(0.99)), bins=70)
axes[1].axvline(0, linestyle="--", color="black", linewidth=1)
axes[1].set(title="Residual distribution between 1st and 99th percentiles", xlabel="actual - predicted")
plt.tight_layout()

## 날짜별 안정성

In [ ]:
daily_scores = group_scores(predictions, "target_date").sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
daily_scores["rmsle"].plot(marker="o", ax=axes[0])
daily_scores["mae"].plot(marker="o", color="tab:orange", ax=axes[1])
axes[0].set(title="Daily RMSLE", xlabel="", ylabel="RMSLE")
axes[1].set(title="Daily MAE", xlabel="", ylabel="MAE")
plt.tight_layout()

## 상품군별 오차

RMSLE는 저판매 상품군의 상대적 오차에 민감하고, MAE는 판매 규모가 큰 상품군의 절대 오차에 민감합니다. 두 지표를 함께 확인합니다.

In [ ]:
family_scores = group_scores(predictions, "family")
family_scores.head(15)

In [ ]:
worst_rmsle = family_scores.head(12).sort_values("rmsle")
worst_mae = family_scores.nlargest(12, "mae").sort_values("mae")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
worst_rmsle["rmsle"].plot.barh(ax=axes[0], title="Highest family RMSLE")
worst_mae["mae"].plot.barh(ax=axes[1], color="tab:orange", title="Highest family MAE")
axes[0].set(xlabel="RMSLE", ylabel="")
axes[1].set(xlabel="MAE", ylabel="")
plt.tight_layout()

## 점포별 오차

In [ ]:
store_scores = group_scores(predictions, "store_nbr")
store_scores.head(15)

## 큰 절대오차 사례

In [ ]:
top_errors(predictions)

## 변수 중요도

유가·연도·주차처럼 시간과 함께 움직이는 변수는 실제 영향과 시간 추세의 대리효과가 섞일 수 있습니다. 중요도만으로 인과관계를 주장하지 않고 후속 제거 실험으로 확인합니다.

In [ ]:
top_importance = importance.head(20).sort_values("importance")
plt.figure(figsize=(9, 7))
plt.barh(top_importance["feature"], top_importance["importance"])
plt.title("LightGBM feature importance")
plt.xlabel("split importance")
plt.tight_layout()

## 다음 실험 기준

1. 날짜별 오차가 급증한 구간의 공휴일·프로모션 여부를 확인합니다.
2. 상품군별 RMSLE와 MAE를 함께 보고 개선 대상을 정합니다.
3. 유가와 절대 날짜 변수를 제거한 모델을 재학습해 시간 대리효과를 확인합니다.
4. LightGBM 설정을 확정한 뒤 동일한 시간 분할로 CatBoost를 비교합니다.